# Feature Engineering & Exploratory Data Analysis

This notebook performs feature engineering on the cleaned Open Food Facts data and conducts comprehensive exploratory data analysis.

## Objectives:
- Load cleaned data from previous step
- Create features for recommendation system
- Vectorize ingredients and product descriptions
- Generate allergen and dietary flags
- Perform comprehensive EDA with visualizations
- Analyze data patterns and distributions

In [ ]:
# Import required libraries
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.ml.feature import Tokenizer, StopWordsRemover, CountVectorizer, IDF
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml import Pipeline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import re
from collections import Counter
from wordcloud import WordCloud
from datetime import datetime

# Set up plotting
plt.style.use('default')
sns.set_palette('husl')
%matplotlib inline

# Initialize Spark Session
spark = SparkSession.builder \
    .appName("FeatureEngineeringEDA_Filtered") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.driver.memory", "4g") \
    .config("spark.driver.maxResultSize", "2g") \
    .getOrCreate()

print(f"Spark Version: {spark.version}")
print(f"Spark UI: {spark.sparkContext.uiWebUrl}")

# Load filtered cleaned data
data_path = "../data/cleaned_food_data_filtered.parquet"
if os.path.exists(data_path):
    df = spark.read.parquet(data_path)
    print(f"Loaded filtered cleaned data: {df.count():,} rows x {len(df.columns)} columns")
    print("Dataset contains products from: Morocco, Spain, and Egypt")
else:
    # Fallback to regular cleaned data
    fallback_path = "../data/cleaned_food_data.parquet"
    if os.path.exists(fallback_path):
        print("Filtered data not found, loading regular cleaned data...")
        df = spark.read.parquet(fallback_path)
        print(f"Loaded cleaned data: {df.count():,} rows x {len(df.columns)} columns")
    else:
        print("Cleaned data not found. Please run data_ingestion_cleaning.ipynb first.")
        raise FileNotFoundError("Cleaned data file not found")

# Load filtering metadata if available
metadata_path = "../data/filtering_metadata.json"
if os.path.exists(metadata_path):
    import json
    with open(metadata_path, 'r') as f:
        filtering_metadata = json.load(f)
    print(f"\nFiltering info: {filtering_metadata['note']}")
    print(f"Target countries: {', '.join(filtering_metadata['filtered_countries'])}")

# Feature engineering: ingredient vectorization, allergen flags, etc.
# ... code here ...
# EDA: distributions, correlations, visualizations
# ... code here ...

In [ ]:
# Basic data overview
print("=== DATA OVERVIEW ===")
print(f"Dataset shape: {df.count():,} rows x {len(df.columns)} columns")
print("\nColumn names:")
for i, col in enumerate(df.columns):
    print(f"{i+1:2d}. {col}")

# Show sample data
print("\n=== SAMPLE DATA ===")
df.show(5, truncate=False)

# Check data types
print("\n=== SCHEMA ===")
df.printSchema()

In [ ]:
# Feature Engineering - Ingredient Processing
print("=== INGREDIENT PROCESSING ===")

# 1. Clean and tokenize ingredients
if 'ingredients_text' in df.columns:
    # Remove special characters and normalize
    df_features = df.withColumn(
        'ingredients_clean',
        regexp_replace(
            regexp_replace(col('ingredients_text'), r'[^a-zA-Z\s,]', ''),
            r'\s+', ' '
        )
    )
    
    # Tokenize ingredients
    tokenizer = Tokenizer(inputCol="ingredients_clean", outputCol="ingredients_tokens")
    df_features = tokenizer.transform(df_features)
    
    # Remove stop words
    stop_words_remover = StopWordsRemover(
        inputCol="ingredients_tokens", 
        outputCol="ingredients_filtered",
        stopWords=StopWordsRemover.loadDefaultStopWords("english") + 
                  ['water', 'salt', 'sugar', 'contains', 'may', 'contain']
    )
    df_features = stop_words_remover.transform(df_features)
    
    print("Ingredient tokenization completed")
    
    # Show sample tokenized ingredients
    sample_ingredients = df_features.select('product_name', 'ingredients_text', 'ingredients_filtered').limit(3)
    for row in sample_ingredients.collect():
        print(f"\nProduct: {row['product_name']}")
        print(f"Original: {row['ingredients_text'][:100]}...")
        print(f"Filtered: {row['ingredients_filtered'][:10]}")
else:
    df_features = df
    print("No ingredients_text column found")

In [ ]:
# 2. Process categories
print("\n=== CATEGORY PROCESSING ===")

if 'categories_en' in df_features.columns:
    # Extract main categories
    df_features = df_features.withColumn(
        'main_category',
        when(col('categories_en').isNotNull(),
             split(col('categories_en'), ',')[0])
        .otherwise('unknown')
    )
    
    # Count category frequency
    category_counts = df_features.groupBy('main_category').count().orderBy(desc('count'))
    print("\nTop 20 main categories:")
    category_counts.show(20, truncate=False)
    
    # Create category features
    df_features = df_features.withColumn(
        'categories_tokens',
        split(regexp_replace(col('categories_en'), r'[^a-zA-Z\s,]', ''), ',')
    )
    
    print("Category processing completed")
else:
    print("No categories_en column found")

In [ ]:
# 3. Create allergen flags
print("\n=== ALLERGEN PROCESSING ===")

common_allergens = ['gluten', 'milk', 'eggs', 'nuts', 'peanuts', 'soy', 'fish', 'shellfish', 'sesame']

if 'allergens_en' in df_features.columns:
    for allergen in common_allergens:
        df_features = df_features.withColumn(
            f'contains_{allergen}',
            when(col('allergens_en').contains(allergen), 1).otherwise(0)
        )
    
    # Count allergen occurrences
    allergen_stats = df_features.agg(
        *[sum(f'contains_{allergen}').alias(f'{allergen}_count') for allergen in common_allergens]
    ).collect()[0]
    
    print("Allergen distribution:")
    for allergen in common_allergens:
        count = allergen_stats[f'{allergen}_count']
        print(f"{allergen:12}: {count:8,} products")
    
    print("Allergen flags created")
else:
    print("No allergens_en column found")

In [ ]:
# 4. Create nutrition-based features
print("\n=== NUTRITION FEATURES ===")

# Create nutrition quality score
nutrition_cols = ['proteins_100g', 'fiber_100g', 'fat_100g', 'sugars_100g', 'salt_100g']
available_nutrition = [col for col in nutrition_cols if col in df_features.columns]

if available_nutrition:
    # Normalize nutrition values and create composite score
    for col_name in available_nutrition:
        df_features = df_features.withColumn(
            f'{col_name}_norm',
            coalesce(col(col_name), lit(0))
        )
    
    # Create healthy score (higher protein and fiber, lower sugar and salt)
    healthy_score_expr = (
        coalesce(col('proteins_100g'), lit(0)) * 0.3 +
        coalesce(col('fiber_100g'), lit(0)) * 0.3 -
        coalesce(col('sugars_100g'), lit(0)) * 0.2 -
        coalesce(col('salt_100g'), lit(0)) * 0.2
    )
    
    df_features = df_features.withColumn('healthy_score', healthy_score_expr)
    
    print(f"Nutrition features created for: {available_nutrition}")
    
    # Show nutrition statistics
    nutrition_stats = df_features.select(
        *[mean(col).alias(f'{col}_mean') for col in available_nutrition if col in df_features.columns],
        mean('healthy_score').alias('healthy_score_mean')
    ).collect()[0]
    
    print("\nNutrition averages:")
    for col in available_nutrition:
        if f'{col}_mean' in nutrition_stats.asDict():
            avg_val = nutrition_stats[f'{col}_mean']
            print(f"{col:20}: {avg_val:8.2f}")
    print(f"{'healthy_score':20}: {nutrition_stats['healthy_score_mean']:8.2f}")
else:
    print("No nutrition columns found")

In [ ]:
# 5. Create text vectorization features
print("\n=== TEXT VECTORIZATION ===")

# Combine product name and ingredients for content-based similarity
df_features = df_features.withColumn(
    'content_text',
    concat_ws(' ',
        coalesce(col('product_name'), lit('')),
        coalesce(col('generic_name'), lit('')),
        coalesce(col('categories_en'), lit(''))
    )
)

# Tokenize combined content
content_tokenizer = Tokenizer(inputCol="content_text", outputCol="content_tokens")
df_features = content_tokenizer.transform(df_features)

# Remove stop words from content
content_stop_remover = StopWordsRemover(
    inputCol="content_tokens", 
    outputCol="content_filtered"
)
df_features = content_stop_remover.transform(df_features)

print("Text vectorization features created")

# Show sample content processing
sample_content = df_features.select('product_name', 'content_text', 'content_filtered').limit(3)
for row in sample_content.collect():
    print(f"\nProduct: {row['product_name']}")
    print(f"Content: {row['content_text'][:100]}...")
    print(f"Filtered tokens: {row['content_filtered'][:10]}")

In [ ]:
# EXPLORATORY DATA ANALYSIS
print("\n" + "="*50)
print("=== EXPLORATORY DATA ANALYSIS ===")
print("="*50)

# 1. Basic statistics
print("\n1. BASIC STATISTICS")
print("-" * 30)

total_products = df_features.count()
print(f"Total products: {total_products:,}")

# Completeness analysis
if 'completeness_score' in df_features.columns:
    completeness_stats = df_features.agg(
        mean('completeness_score').alias('avg_completeness'),
        min('completeness_score').alias('min_completeness'),
        max('completeness_score').alias('max_completeness')
    ).collect()[0]
    
    print(f"Average completeness: {completeness_stats['avg_completeness']:.1f}%")
    print(f"Completeness range: {completeness_stats['min_completeness']:.0f}% - {completeness_stats['max_completeness']:.0f}%")

# Country distribution
if 'primary_country' in df_features.columns:
    country_dist = df_features.groupBy('primary_country').count().orderBy(desc('count'))
    print("\nTop 10 countries:")
    country_dist.show(10, truncate=False)

In [ ]:
# 2. Nutrition Score Analysis
print("\n2. NUTRITION SCORE ANALYSIS")
print("-" * 35)

# Nutri-Score distribution
if 'nutriscore_grade' in df_features.columns:
    nutriscore_dist = df_features.groupBy('nutriscore_grade').count().orderBy('nutriscore_grade')
    print("\nNutri-Score distribution:")
    nutriscore_dist.show()
    
    # Convert to pandas for visualization
    nutriscore_pd = nutriscore_dist.toPandas()
    
    if not nutriscore_pd.empty:
        plt.figure(figsize=(12, 8))
        
        # Nutri-Score bar chart
        plt.subplot(2, 2, 1)
        nutriscore_pd = nutriscore_pd.dropna()
        colors = ['#2E8B57', '#90EE90', '#FFD700', '#FF8C00', '#FF6347']
        plt.bar(nutriscore_pd['nutriscore_grade'], nutriscore_pd['count'], color=colors[:len(nutriscore_pd)])
        plt.title('Nutri-Score Distribution')
        plt.xlabel('Nutri-Score Grade')
        plt.ylabel('Number of Products')
        
        # Eco-Score distribution if available
        if 'ecoscore_grade' in df_features.columns:
            ecoscore_dist = df_features.groupBy('ecoscore_grade').count().orderBy('ecoscore_grade')
            ecoscore_pd = ecoscore_dist.toPandas().dropna()
            
            if not ecoscore_pd.empty:
                plt.subplot(2, 2, 2)
                plt.bar(ecoscore_pd['ecoscore_grade'], ecoscore_pd['count'], color='lightblue')
                plt.title('Eco-Score Distribution')
                plt.xlabel('Eco-Score Grade')
                plt.ylabel('Number of Products')
        
        plt.tight_layout()
        plt.show()
else:
    print("No nutriscore_grade column found")

In [ ]:
# 3. Category Analysis
print("\n3. CATEGORY ANALYSIS")
print("-" * 25)

if 'main_category' in df_features.columns:
    # Get top categories
    top_categories = df_features.groupBy('main_category').count().orderBy(desc('count')).limit(15)
    top_categories_pd = top_categories.toPandas()
    
    # Create category visualization
    plt.figure(figsize=(15, 10))
    
    # Horizontal bar chart for top categories
    plt.subplot(2, 1, 1)
    plt.barh(range(len(top_categories_pd)), top_categories_pd['count'])
    plt.yticks(range(len(top_categories_pd)), top_categories_pd['main_category'])
    plt.xlabel('Number of Products')
    plt.title('Top 15 Product Categories')
    plt.gca().invert_yaxis()
    
    # Category pie chart for top 10
    plt.subplot(2, 1, 2)
    top_10_categories = top_categories_pd.head(10)
    plt.pie(top_10_categories['count'], labels=top_10_categories['main_category'], autopct='%1.1f%%')
    plt.title('Top 10 Categories Distribution')
    
    plt.tight_layout()
    plt.show()
    
    print("\nTop 10 categories:")
    top_categories.show(10, truncate=False)
else:
    print("No main_category column found")

In [ ]:
# 4. Nutritional Value Distributions
print("\n4. NUTRITIONAL VALUE DISTRIBUTIONS")
print("-" * 40)

nutrition_cols = ['energy_100g', 'proteins_100g', 'carbohydrates_100g', 'fat_100g', 'fiber_100g', 'sugars_100g', 'salt_100g']
available_nutrition = [col for col in nutrition_cols if col in df_features.columns]

if available_nutrition:
    # Sample data for visualization (10% sample)
    sample_df = df_features.sample(0.1).select(*available_nutrition).toPandas()
    sample_df = sample_df.dropna()
    
    if not sample_df.empty:
        # Create nutrition distribution plots
        n_cols = 3
        n_rows = (len(available_nutrition) + n_cols - 1) // n_cols
        
        plt.figure(figsize=(15, 4 * n_rows))
        
        for i, col in enumerate(available_nutrition):
            plt.subplot(n_rows, n_cols, i + 1)
            
            # Remove outliers for better visualization
            Q1 = sample_df[col].quantile(0.25)
            Q3 = sample_df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower_bound = Q1 - 1.5 * IQR
            upper_bound = Q3 + 1.5 * IQR
            
            filtered_data = sample_df[(sample_df[col] >= lower_bound) & (sample_df[col] <= upper_bound)][col]
            
            plt.hist(filtered_data, bins=30, alpha=0.7, edgecolor='black')
            plt.title(f'{col} Distribution (outliers removed)')
            plt.xlabel(col.replace('_', ' ').title())
            plt.ylabel('Frequency')
            plt.grid(axis='y', alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        # Nutrition correlation matrix
        plt.figure(figsize=(10, 8))
        correlation_matrix = sample_df[available_nutrition].corr()
        sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, square=True)
        plt.title('Nutritional Values Correlation Matrix')
        plt.tight_layout()
        plt.show()
        
        print("\nNutrition statistics:")
        print(sample_df[available_nutrition].describe())
else:
    print("No nutrition columns found")

In [ ]:
# 5. Allergen Analysis
print("\n5. ALLERGEN ANALYSIS")
print("-" * 25)

allergen_cols = [col for col in df_features.columns if col.startswith('contains_')]

if allergen_cols:
    # Calculate allergen percentages
    total_products = df_features.count()
    allergen_stats = []
    
    for col in allergen_cols:
        allergen_name = col.replace('contains_', '')
        count = df_features.agg(sum(col).alias('count')).collect()[0]['count']
        percentage = (count / total_products) * 100
        allergen_stats.append((allergen_name, count, percentage))
    
    # Sort by percentage
    allergen_stats.sort(key=lambda x: x[2], reverse=True)
    
    # Create allergen visualization
    allergen_names = [x[0] for x in allergen_stats]
    allergen_percentages = [x[2] for x in allergen_stats]
    
    plt.figure(figsize=(12, 6))
    
    plt.subplot(1, 2, 1)
    bars = plt.bar(allergen_names, allergen_percentages, color='lightcoral')
    plt.title('Allergen Prevalence in Products')
    plt.xlabel('Allergens')
    plt.ylabel('Percentage of Products (%)')
    plt.xticks(rotation=45)
    plt.grid(axis='y', alpha=0.3)
    
    # Add percentage labels on bars
    for bar, pct in zip(bars, allergen_percentages):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                f'{pct:.1f}%', ha='center', va='bottom')
    
    # Allergen co-occurrence heatmap
    if len(allergen_cols) > 1:
        allergen_sample = df_features.select(*allergen_cols).sample(0.1).toPandas()
        plt.subplot(1, 2, 2)
        allergen_corr = allergen_sample.corr()
        allergen_corr.index = [col.replace('contains_', '') for col in allergen_corr.index]
        allergen_corr.columns = [col.replace('contains_', '') for col in allergen_corr.columns]
        sns.heatmap(allergen_corr, annot=True, cmap='Reds', square=True)
        plt.title('Allergen Co-occurrence')
    
    plt.tight_layout()
    plt.show()
    
    print("\nAllergen statistics:")
    for name, count, pct in allergen_stats:
        print(f"{name:12}: {count:8,} products ({pct:5.2f}%)")
else:
    print("No allergen columns found")

In [ ]:
# 6. Ingredient Word Cloud Analysis
print("\n6. INGREDIENT ANALYSIS")
print("-" * 30)

if 'ingredients_filtered' in df_features.columns:
    # Collect ingredient tokens
    ingredients_sample = df_features.select('ingredients_filtered').sample(0.05).collect()
    all_ingredients = []
    
    for row in ingredients_sample:
        if row['ingredients_filtered']:
            all_ingredients.extend(row['ingredients_filtered'])
    
    if all_ingredients:
        # Count ingredient frequency
        ingredient_counts = Counter(all_ingredients)
        top_ingredients = ingredient_counts.most_common(50)
        
        print("\nTop 20 most common ingredients:")
        for ingredient, count in top_ingredients[:20]:
            print(f"{ingredient:20}: {count:5,} times")
        
        # Create word cloud
        try:
            wordcloud_text = ' '.join([ingredient for ingredient, _ in top_ingredients])
            wordcloud = WordCloud(
                width=800, height=400, 
                background_color='white',
                max_words=100,
                colormap='viridis'
            ).generate(wordcloud_text)
            
            plt.figure(figsize=(12, 6))
            plt.imshow(wordcloud, interpolation='bilinear')
            plt.axis('off')
            plt.title('Most Common Ingredients Word Cloud')
            plt.tight_layout()
            plt.show()
        except Exception as e:
            print(f"Could not generate word cloud: {e}")
        
        # Ingredient frequency bar chart
        top_20_ingredients = top_ingredients[:20]
        ingredients_names = [x[0] for x in top_20_ingredients]
        ingredients_counts = [x[1] for x in top_20_ingredients]
        
        plt.figure(figsize=(12, 8))
        plt.barh(range(len(ingredients_names)), ingredients_counts)
        plt.yticks(range(len(ingredients_names)), ingredients_names)
        plt.xlabel('Frequency')
        plt.title('Top 20 Most Common Ingredients')
        plt.gca().invert_yaxis()
        plt.tight_layout()
        plt.show()
else:
    print("No ingredients_filtered column found")

In [ ]:
# 7. Feature Importance for Recommendation
print("\n7. FEATURE IMPORTANCE ANALYSIS")
print("-" * 40)

# Analyze which features are most complete and useful
feature_completeness = []

for col in df_features.columns:
    if col not in ['processed_at', 'completeness_score']:
        non_null_count = df_features.filter(col(col).isNotNull() & (col(col) != "")).count()
        completeness_pct = (non_null_count / total_products) * 100
        feature_completeness.append((col, non_null_count, completeness_pct))

# Sort by completeness
feature_completeness.sort(key=lambda x: x[2], reverse=True)

print("\nFeature completeness ranking:")
print(f"{'Feature':<30} {'Non-null Count':<15} {'Completeness %':<15}")
print("-" * 60)
for feature, count, pct in feature_completeness[:20]:
    print(f"{feature:<30} {count:<15,} {pct:<15.1f}%")

# Visualize feature completeness
plt.figure(figsize=(12, 8))
top_features = feature_completeness[:15]
feature_names = [x[0] for x in top_features]
completeness_pcts = [x[2] for x in top_features]

plt.barh(range(len(feature_names)), completeness_pcts, color='lightgreen')
plt.yticks(range(len(feature_names)), feature_names)
plt.xlabel('Completeness Percentage (%)')
plt.title('Top 15 Features by Data Completeness')
plt.gca().invert_yaxis()
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Save engineered features for filtered dataset
print("\n" + "="*50)
print("=== SAVING ENGINEERED FEATURES (FILTERED) ===")
print("="*50)

# Select final features for recommendation system
final_features = [
    'code', 'product_name', 'generic_name', 'brands', 'countries_en', 'primary_country',
    'categories_en', 'main_category', 'ingredients_text', 'ingredients_filtered',
    'allergens_en', 'nutriscore_grade', 'ecoscore_grade', 'nova_group',
    'energy_100g', 'proteins_100g', 'carbohydrates_100g', 'fat_100g',
    'fiber_100g', 'sugars_100g', 'salt_100g', 'sodium_100g',
    'packaging', 'image_url', 'image_front_url',
    'completeness_score', 'content_text', 'content_filtered'
]

# Add allergen flags if they exist
for allergen in ['gluten', 'milk', 'eggs', 'nuts', 'peanuts', 'soy', 'fish', 'shellfish', 'sesame']:
    if f'contains_{allergen}' in df_features.columns:
        final_features.append(f'contains_{allergen}')

# Add nutrition scores if they exist
if 'healthy_score' in df_features.columns:
    final_features.append('healthy_score')

# Select only existing columns
existing_features = [col for col in final_features if col in df_features.columns]
df_final = df_features.select(*existing_features)

print(f"Selected {len(existing_features)} features for recommendation system")
print(f"Final dataset shape: {df_final.count():,} rows x {len(existing_features)} columns")
print(f"Target markets: Morocco, Spain, and Egypt")

# Save engineered features for filtered data
features_output_path = "../data/engineered_features_filtered.parquet"
df_final.coalesce(5).write \
    .mode('overwrite') \
    .option('compression', 'snappy') \
    .parquet(features_output_path)

print(f"\nFiltered engineered features saved to: {features_output_path}")

# Save feature metadata for filtered dataset
feature_metadata = {
    'total_features': len(existing_features),
    'total_records': df_final.count(),
    'feature_list': existing_features,
    'processing_date': datetime.now().isoformat(),
    'target_countries': ['Morocco', 'Spain', 'Egypt'],
    'dataset_type': 'filtered',
    'note': 'Engineered features for products from Morocco, Spain, and Egypt only'
}

with open('../data/feature_metadata_filtered.json', 'w') as f:
    json.dump(feature_metadata, f, indent=2)

print("Filtered feature metadata saved to: ../data/feature_metadata_filtered.json")

# Final summary
print("\n=== FEATURE ENGINEERING SUMMARY (FILTERED) ===")
print(f"✅ Processed {df_final.count():,} products from target countries")
print(f"✅ Created {len(existing_features)} features")
print(f"✅ Ingredient tokenization completed")
print(f"✅ Category processing completed")
print(f"✅ Allergen flags created")
print(f"✅ Nutrition features engineered")
print(f"✅ Text vectorization features prepared")
print(f"✅ Comprehensive EDA completed")
print(f"✅ Dataset optimized for Morocco, Spain, and Egypt markets")
print(f"✅ Efficient size for model training and deployment")

# Stop Spark session
spark.stop()
print("\nSpark session stopped. Feature engineering and EDA completed successfully!")
print("\n🌍 Ready for recommendation system training with regional focus!")